# LearningDNA Hyperparameter Tuning

This notebook **reuses the existing LearningDNA model, prepared parquet data, vocabularies, dataset class, and collate function** from `EdNet_KT1_Training.ipynb`. It does not run the upstream preprocessing or feature-generation notebooks.

Purpose: compare a small set of **previously untried** learning-rate, dropout, and weight-decay configurations using one complete pass over the same selected processed parquet parts. Each experiment starts with a fresh model and optimizer. Results and checkpoints are saved incrementally so the run can continue after interruption.

> Screening uses the first `TUNING_MAX_PARTS` processed parquet files by default. This is one complete pass over that selected subset, not a full-dataset epoch. Confirm finalists with longer/full training before choosing the final configuration.


In [1]:
import os
import math
import time
import json
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import IterableDataset, DataLoader

torch.set_num_threads(1)
torch.set_num_interop_threads(1)

print(
    "Torch CPU threads:",
    torch.get_num_threads()
)

print(
    "Torch interop threads:",
    torch.get_num_interop_threads()
)

Torch CPU threads: 1
Torch interop threads: 1


In [2]:
# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROCESSED_DIR = Path(
    r"D:\Lokesh files\Projects\LearningDNA\processed\kt1_processed"
)

VECTORS_DIR = Path(
    r"D:\Lokesh files\Projects\LearningDNA\processed\kt1_vectors"
)

MODEL_DIR = Path(
    r"D:\Lokesh files\Projects\LearningDNA\models\learningdna_v2"
)

MODEL_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = MODEL_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

    props = torch.cuda.get_device_properties(0)

    print(
        "VRAM:",
        round(props.total_memory / 1024**3, 2),
        "GB"
    )

PyTorch: 2.8.0+cu129
Device: cuda
GPU: NVIDIA GeForce MX450
VRAM: 2.0 GB


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)

Seed: 42


In [4]:

# ============================================================
# FEATURE DIMENSIONS — LEARNINGDNA V2
# ============================================================

QUESTION_DIM = 64
SKILL_DIM = 32
CORRECTNESS_DIM = 8
ELAPSED_DIM = 16
ATTEMPT_DIM = 8
TIME_DELTA_DIM = 8

# ============================================================
# TRANSFORMER
# ============================================================

MODEL_DIM = 128
NUM_HEADS = 8
NUM_LAYERS = 2
FF_DIM = 256
DROPOUT = 0.15  # default; each experiment overrides this

# ============================================================
# SEQUENCE
# ============================================================

SEQ_LEN = 100
STRIDE = 50

# ============================================================
# TRAINING
# ============================================================

BATCH_SIZE = 64
NUM_WORKERS = 0
GRAD_ACCUM_STEPS = 1

DROPOUT = 0.15  # default; each experiment overrides this5
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4

MAX_EPOCHS = 20

# ============================================================
# DATA SPLIT — KEEP EXISTING METHOD
# ============================================================

VAL_MOD = 10

# ============================================================
# CHECKPOINTING
# ============================================================

SAVE_EVERY_STEPS = 500
SAVE_EVERY_MINUTES = 15

# ============================================================
# PILOT
# ============================================================

PILOT_PARTS = 3

# ============================================================
# PERFORMANCE
# ============================================================

USE_AMP = DEVICE.type == "cuda"

print("LearningDNA V2 configuration loaded.")
print("Model dimension:", MODEL_DIM)
print("Transformer layers:", NUM_LAYERS)
print("Attention heads:", NUM_HEADS)
print("Sequence length:", SEQ_LEN)
print("Stride:", STRIDE)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Maximum epochs:", MAX_EPOCHS)
print("Checkpoint interval:", SAVE_EVERY_MINUTES)
print("AMP:", USE_AMP)

LearningDNA V2 configuration loaded.
Model dimension: 128
Transformer layers: 2
Attention heads: 8
Sequence length: 100
Stride: 50
Batch size: 64
Learning rate: 0.0002
Maximum epochs: 20
Checkpoint interval: 15
AMP: True


In [5]:
QUESTION_VOCAB_FILE = (
    VECTORS_DIR / "question_vocab.pkl"
)

SKILL_VOCAB_FILE = (
    VECTORS_DIR / "skill_vocab.pkl"
)

with open(QUESTION_VOCAB_FILE, "rb") as f:
    question_to_idx = pickle.load(f)

with open(SKILL_VOCAB_FILE, "rb") as f:
    skill_to_idx = pickle.load(f)

NUM_QUESTIONS = len(question_to_idx)
NUM_SKILLS = len(skill_to_idx)

print("Number of questions:", NUM_QUESTIONS)
print("Number of skills:", NUM_SKILLS)

Number of questions: 12284
Number of skills: 190


In [6]:
QUESTIONS_FILE = Path(
    r"D:\Lokesh files\Projects\LearningDNA\contents\questions.csv"
)

questions_df = pd.read_csv(
    QUESTIONS_FILE
)

print(
    "questions.csv rows:",
    len(questions_df)
)

print(
    "Columns:",
    questions_df.columns.tolist()
)

question_skill_map = {}

for _, row in questions_df.iterrows():

    qid = str(row["question_id"]).strip()

    tags = row["tags"]

    if pd.isna(tags):
        question_skill_map[qid] = []
        continue

    skills = []

    for tag in str(tags).split(";"):

        tag = tag.strip()

        if tag in skill_to_idx:
            skills.append(
                skill_to_idx[tag]
            )

    question_skill_map[qid] = skills

print(
    "Question-skill mappings:",
    len(question_skill_map)
)

questions.csv rows: 13169
Columns: ['question_id', 'bundle_id', 'explanation_id', 'correct_answer', 'part', 'tags', 'deployed_at']
Question-skill mappings: 13169


In [7]:
# ============================================================
# PRECOMPUTE QUESTION → SKILL TENSOR
# ============================================================

MAX_SKILLS = 4

question_skill_tensor = torch.full(
    (
        NUM_QUESTIONS,
        MAX_SKILLS
    ),
    -1,
    dtype=torch.long
)

for question_id, question_idx in question_to_idx.items():

    skills = question_skill_map.get(
        question_id,
        []
    )

    for j, skill_idx in enumerate(
        skills[:MAX_SKILLS]
    ):
        question_skill_tensor[
            question_idx,
            j
        ] = skill_idx

print(
    "Question-skill tensor shape:",
    tuple(question_skill_tensor.shape)
)

print(
    "Example q1 index:",
    question_to_idx.get("q1")
)

q1_idx = question_to_idx.get("q1")

if q1_idx is not None:
    print(
        "q1 skills:",
        question_skill_tensor[q1_idx].tolist()
    )

Question-skill tensor shape: (12284, 4)
Example q1 index: 0
q1 skills: [2, 100, 88, 91]


In [8]:
class LearningDNA(nn.Module):

    def __init__(
        self,
        num_questions,
        num_skills,
        question_dim=64,
        skill_dim=32,
        correctness_dim=8,
        elapsed_dim=16,
        attempt_dim=8,
        time_delta_dim=8,
        model_dim=128,
        num_heads=4,
        num_layers=2,
        ff_dim=256,
        dropout=0.1,
        max_seq_len=64
    ):
        super().__init__()

        self.model_dim = model_dim

        # ----------------------------------------------------
        # Question embedding
        # ----------------------------------------------------

        self.question_embedding = nn.Embedding(
            num_questions,
            question_dim
        )

        # 0 = incorrect
        # 1 = correct
        # 2 = missing

        self.correctness_embedding = nn.Embedding(
            3,
            correctness_dim
        )

        # Attempt 1...256
        self.attempt_embedding = nn.Embedding(
            256,
            attempt_dim
        )

        # ----------------------------------------------------
        # Skill embedding
        # ----------------------------------------------------

        self.skill_embedding = nn.Embedding(
            num_skills,
            skill_dim
        )

        # ----------------------------------------------------
        # Continuous features
        # ----------------------------------------------------

        self.elapsed_projection = nn.Sequential(
            nn.Linear(1, elapsed_dim),
            nn.ReLU()
        )

        self.time_delta_projection = nn.Sequential(
            nn.Linear(1, time_delta_dim),
            nn.ReLU()
        )

        # ----------------------------------------------------
        # Feature fusion
        # ----------------------------------------------------

        input_dim = (
            question_dim
            + skill_dim
            + correctness_dim
            + elapsed_dim
            + attempt_dim
            + time_delta_dim
        )

        self.feature_projection = nn.Linear(
            input_dim,
            model_dim
        )

        # ----------------------------------------------------
        # Positional embedding
        # ----------------------------------------------------

        self.position_embedding = nn.Embedding(
            max_seq_len,
            model_dim
        )

        # ----------------------------------------------------
        # Transformer
        # ----------------------------------------------------

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # ----------------------------------------------------
        # Prediction head
        # ----------------------------------------------------

        self.prediction_head = nn.Sequential(
            nn.Linear(
                model_dim + question_dim,
                128
            ),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        question_idx,
        skill_idx,
        correctness_idx,
        elapsed,
        attempt_idx,
        time_delta,
        target_question_idx
    ):

        # ----------------------------------------------------
        # Question
        # ----------------------------------------------------

        q = self.question_embedding(
            question_idx
        )

        # ----------------------------------------------------
        # Skills
        # ----------------------------------------------------

        skill_emb = self.skill_embedding(
            skill_idx.clamp(min=0)
        )

        valid_skill = (
            skill_idx >= 0
        ).unsqueeze(-1)

        skill_emb = (
            skill_emb * valid_skill
        )

        skill_count = (
            valid_skill.sum(dim=2)
            .clamp(min=1)
        )

        skill_emb = (
            skill_emb.sum(dim=2)
            / skill_count
        )

        # ----------------------------------------------------
        # Correctness
        # ----------------------------------------------------

        c = self.correctness_embedding(
            correctness_idx
        )

        # ----------------------------------------------------
        # Elapsed time
        # ----------------------------------------------------

        e = self.elapsed_projection(
            elapsed
        )

        # ----------------------------------------------------
        # Attempt
        # ----------------------------------------------------

        a = self.attempt_embedding(
            attempt_idx.clamp(
                min=0,
                max=255
            )
        )

        # ----------------------------------------------------
        # Time delta
        # ----------------------------------------------------

        td = self.time_delta_projection(
            time_delta
        )

        # ----------------------------------------------------
        # Combine features
        # ----------------------------------------------------

        x = torch.cat(
            [
                q,
                skill_emb,
                c,
                e,
                a,
                td
            ],
            dim=-1
        )

        x = self.feature_projection(
            x
        )

        # ----------------------------------------------------
        # Position
        # ----------------------------------------------------

        positions = torch.arange(
            x.size(1),
            device=x.device
        )

        x = (
            x
            + self.position_embedding(
                positions
            )
        )

        x = self.dropout(x)

        # ----------------------------------------------------
        # Causal attention mask
        # ----------------------------------------------------

        seq_len = x.size(1)

        causal_mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                device=x.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        # ----------------------------------------------------
        # Transformer
        # ----------------------------------------------------

        h = self.transformer(
            x,
            mask=causal_mask
        )

        # ----------------------------------------------------
        # Next question
        # ----------------------------------------------------

        next_q = self.question_embedding(
            target_question_idx
        )

        # ----------------------------------------------------
        # Prediction
        # ----------------------------------------------------

        combined = torch.cat(
            [
                h,
                next_q
            ],
            dim=-1
        )

        logits = self.prediction_head(
            combined
        ).squeeze(-1)

        return logits, h

In [9]:
PART_FILES = sorted(
    PROCESSED_DIR.glob("part_*.parquet")
)

print(
    "Parquet parts:",
    len(PART_FILES)
)

TRAIN_COLUMNS = [
    "user_id",
    "question_id",
    "correctness",
    "answer_missing",
    "log_elapsed_time",
    "attempt_index",
    "log_time_delta"
]

Parquet parts: 15687


In [10]:
# ============================================================
# KT SEQUENCE DATASET - PART-LEVEL RESUMABLE
# ============================================================

class KTSequenceDataset(IterableDataset):

    def __init__(
        self,
        part_files,
        train=True,
        seq_len=64,
        stride=32,
        max_parts=None,
        start_part=0
    ):
        self.part_files = part_files
        self.train = train
        self.seq_len = seq_len
        self.stride = stride
        self.max_parts = max_parts
        self.start_part = start_part

    def student_is_train(self, user_id):
        return int(user_id) % VAL_MOD != 0

    def make_samples(self, group, part_index):

        question_ids = (
            group["question_id"]
            .astype(str)
            .to_numpy()
        )

        correctness = group[
            "correctness"
        ].to_numpy(dtype=np.int64)

        missing = group[
            "answer_missing"
        ].to_numpy(dtype=np.int64)

        elapsed = group[
            "log_elapsed_time"
        ].to_numpy(dtype=np.float32)

        attempt = group[
            "attempt_index"
        ].to_numpy(dtype=np.int64)

        time_delta = group[
            "log_time_delta"
        ].to_numpy(dtype=np.float32)

        n = len(group)

        if n < 2:
            return

        start = 0

        while start < n - 1:

            end = min(
                start + self.seq_len,
                n - 1
            )

            if end <= start:
                break

            current_q = question_ids[start:end]

            target_q = question_ids[
                start + 1:end + 1
            ]

            current_correctness = (
                correctness[start:end].copy()
            )

            current_correctness[
                missing[start:end] == 1
            ] = 2

            target_correctness = (
                correctness[start + 1:end + 1].copy()
            )

            target_missing = (
                missing[start + 1:end + 1]
            )

            target_mask = (
                target_missing == 0
            )

            if target_mask.sum() == 0:
                start += self.stride
                continue

            current_q_idx = np.array(
                [
                    question_to_idx.get(q, 0)
                    for q in current_q
                ],
                dtype=np.int64
            )

            target_q_idx = np.array(
                [
                    question_to_idx.get(q, 0)
                    for q in target_q
                ],
                dtype=np.int64
            )

            skill_array = (
                question_skill_tensor[
                    torch.from_numpy(
                        current_q_idx
                    )
                ].numpy()
            )

            attempt_idx = np.clip(
                attempt[start:end] - 1,
                0,
                255
            )

            yield {
                "question_idx": current_q_idx,
                "skill_idx": skill_array,
                "correctness_idx": current_correctness,
                "elapsed": elapsed[start:end],
                "attempt_idx": attempt_idx,
                "time_delta": time_delta[start:end],
                "target_question_idx": target_q_idx,
                "target": target_correctness.astype(
                    np.float32
                ),
                "target_mask": target_mask,
                "part_index": part_index
            }

            start += self.stride

            if end == n - 1:
                break

    def __iter__(self):

        from torch.utils.data import get_worker_info

        files = self.part_files

        if self.max_parts is not None:
            files = files[:self.max_parts]

        worker_info = get_worker_info()

        if worker_info is not None:
            worker_id = worker_info.id
            num_workers = worker_info.num_workers
            files = files[
                worker_id::num_workers
            ]

        files = files[self.start_part:]

        for local_index, part_file in enumerate(files):

            actual_part_index = (
                self.start_part
                + local_index
            )

            df = pd.read_parquet(
                part_file,
                columns=TRAIN_COLUMNS
            )

            df = df.sort_values(
                ["user_id"],
                kind="stable"
            )

            for user_id, group in df.groupby(
                "user_id",
                sort=False
            ):

                if (
                    self.student_is_train(user_id)
                    != self.train
                ):
                    continue

                yield from self.make_samples(
                    group,
                    actual_part_index
                )

            del df

In [11]:
# ============================================================
# COLLATE FUNCTION
# ============================================================

def collate_fn(samples):

    max_len = max(
        len(x["question_idx"])
        for x in samples
    )

    batch_size = len(samples)

    question_idx = torch.zeros(
        batch_size,
        max_len,
        dtype=torch.long
    )

    skill_idx = torch.full(
        (batch_size, max_len, 4),
        -1,
        dtype=torch.long
    )

    correctness_idx = torch.zeros(
        batch_size,
        max_len,
        dtype=torch.long
    )

    elapsed = torch.zeros(
        batch_size,
        max_len,
        1,
        dtype=torch.float32
    )

    attempt_idx = torch.zeros(
        batch_size,
        max_len,
        dtype=torch.long
    )

    time_delta = torch.zeros(
        batch_size,
        max_len,
        1,
        dtype=torch.float32
    )

    target_question_idx = torch.zeros(
        batch_size,
        max_len,
        dtype=torch.long
    )

    target = torch.zeros(
        batch_size,
        max_len,
        dtype=torch.float32
    )

    target_mask = torch.zeros(
        batch_size,
        max_len,
        dtype=torch.bool
    )

    part_indices = torch.zeros(
        batch_size,
        dtype=torch.long
    )

    for i, sample in enumerate(samples):

        length = len(
            sample["question_idx"]
        )

        question_idx[i, :length] = torch.from_numpy(
            sample["question_idx"]
        )

        skill_idx[i, :length] = torch.from_numpy(
            sample["skill_idx"]
        )

        correctness_idx[i, :length] = torch.from_numpy(
            sample["correctness_idx"]
        )

        elapsed[i, :length, 0] = torch.from_numpy(
            sample["elapsed"]
        )

        attempt_idx[i, :length] = torch.from_numpy(
            sample["attempt_idx"]
        )

        time_delta[i, :length, 0] = torch.from_numpy(
            sample["time_delta"]
        )

        target_question_idx[i, :length] = torch.from_numpy(
            sample["target_question_idx"]
        )

        target[i, :length] = torch.from_numpy(
            sample["target"]
        )

        target_mask[i, :length] = torch.from_numpy(
            sample["target_mask"]
        )

        part_indices[i] = sample["part_index"]

    return {
        "question_idx": question_idx,
        "skill_idx": skill_idx,
        "correctness_idx": correctness_idx,
        "elapsed": elapsed,
        "attempt_idx": attempt_idx,
        "time_delta": time_delta,
        "target_question_idx": target_question_idx,
        "target": target,
        "target_mask": target_mask,
        "part_index": part_indices
    }

In [12]:
def move_batch_to_device(batch):

    return {
        key: value.to(
            DEVICE,
            non_blocking=True
        )
        for key, value in batch.items()
    }

## Experiment settings

The configurations below avoid the previously reported pilot/full-run combinations. The search is intentionally small (8 configurations). The training code uses the same model forward pass and binary cross-entropy objective as the existing notebook.

If your previous experiments used additional combinations not listed in the project notes, add them to `EXPERIMENTS` only after checking they are genuinely new.


In [13]:

import csv
import gc
import math
import random
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    mean_squared_error,
    log_loss
)

# ============================================================
# HYPERPARAMETER TUNING V3
# ============================================================

TUNING_DIR = MODEL_DIR / "hyperparameter_tuning_v3"
TUNING_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINTS_DIR = TUNING_DIR / "checkpoints"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = (
    TUNING_DIR / "hyperparameter_search_results.csv"
)

RANKED_CSV = (
    TUNING_DIR / "hyperparameter_search_ranked.csv"
)

# ============================================================
# DATA SELECTION
# ============================================================

TUNING_DATA_FRACTION = 0.50
TUNING_SEED = 42

number_of_parts = max(
    1,
    math.ceil(len(PART_FILES) * TUNING_DATA_FRACTION)
)

# Select a fixed, reproducible subset of parquet parts.
# All experiments use exactly the same selected parts.

rng = random.Random(TUNING_SEED)

TUNING_PART_FILES = sorted(
    rng.sample(PART_FILES, number_of_parts)
)

TUNING_MAX_PARTS = None

# ============================================================
# TRAINING SETTINGS
# ============================================================

TUNING_EPOCHS = 5

SAVE_CHECKPOINT_EVERY_STEPS = 500

# ============================================================
# V3 BASELINE (REFERENCE ONLY)
# ============================================================

V3_BASELINE = {
    "learning_rate": 3e-4,
    "dropout": 0.10,
    "weight_decay": 5e-5,
    "roc_auc": 0.771046,
    "accuracy": 0.735386,
    "rmse": 0.420077
}

# ============================================================
# NEW EXPERIMENTS
# Do not repeat previous H01-H08 configurations.
# ============================================================

EXPERIMENTS = [

    {
        "experiment_id": "N01",
        "learning_rate": 2e-4,
        "dropout": 0.10,
        "weight_decay": 5e-5
    },

    {
        "experiment_id": "N02",
        "learning_rate": 4e-4,
        "dropout": 0.10,
        "weight_decay": 5e-5
    },

    {
        "experiment_id": "N03",
        "learning_rate": 3e-4,
        "dropout": 0.05,
        "weight_decay": 5e-5
    },

    {
        "experiment_id": "N04",
        "learning_rate": 3e-4,
        "dropout": 0.15,
        "weight_decay": 5e-5
    }

]

# ============================================================
# CSV FIELDS
# ============================================================

CSV_FIELDS = [
    "experiment_id",
    "learning_rate",
    "dropout",
    "weight_decay",
    "seed",
    "epoch",
    "epochs_completed",
    "training_steps",
    "training_samples",
    "training_loss",
    "validation_loss",
    "validation_accuracy",
    "roc_auc",
    "pr_auc",
    "rmse",
    "log_loss",
    "training_time_seconds",
    "validation_time_seconds",
    "total_time_seconds",
    "train_parts",
    "validation_parts",
    "checkpoint_path",
    "status",
    "error",
    "completed_at"
]

# ============================================================
# VALIDATION
# ============================================================

if not PART_FILES:
    raise FileNotFoundError(
        f"No processed parquet files found in {PROCESSED_DIR}"
    )

if len(TUNING_PART_FILES) == 0:
    raise ValueError("No tuning parquet parts selected.")

print("=" * 60)
print("LEARNINGDNA V3 — HYPERPARAMETER SEARCH")
print("=" * 60)

print("Device:", DEVICE)
print("Available parquet parts:", len(PART_FILES))
print("Selected parquet parts:", len(TUNING_PART_FILES))
print("Data fraction:", TUNING_DATA_FRACTION)
print("Epochs per experiment:", TUNING_EPOCHS)
print("Number of experiments:", len(EXPERIMENTS))
print("Baseline ROC-AUC:", V3_BASELINE["roc_auc"])
print("Results CSV:", RESULTS_CSV)

print("\nSelected data parts:")
for p in TUNING_PART_FILES:
    print(" -", p.name)


LEARNINGDNA V3 — HYPERPARAMETER SEARCH
Device: cuda
Available parquet parts: 15687
Selected parquet parts: 7844
Data fraction: 0.5
Epochs per experiment: 5
Number of experiments: 4
Baseline ROC-AUC: 0.771046
Results CSV: D:\Lokesh files\Projects\LearningDNA\models\learningdna_v2\hyperparameter_tuning_v3\hyperparameter_search_results.csv

Selected data parts:
 - part_000001.parquet
 - part_000002.parquet
 - part_000003.parquet
 - part_000004.parquet
 - part_000006.parquet
 - part_000007.parquet
 - part_000009.parquet
 - part_000011.parquet
 - part_000013.parquet
 - part_000017.parquet
 - part_000018.parquet
 - part_000019.parquet
 - part_000020.parquet
 - part_000023.parquet
 - part_000025.parquet
 - part_000027.parquet
 - part_000029.parquet
 - part_000030.parquet
 - part_000035.parquet
 - part_000040.parquet
 - part_000042.parquet
 - part_000043.parquet
 - part_000044.parquet
 - part_000046.parquet
 - part_000049.parquet
 - part_000053.parquet
 - part_000056.parquet
 - part_000057.par

## Training and evaluation helpers

The helper functions below follow the existing LearningDNA forward call and masked binary-cross-entropy objective. They add timing, metrics, per-experiment checkpoints, and CSV logging only.

In [14]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def create_tuning_model(dropout):
    # Same LearningDNA architecture as the existing training notebook.
    return LearningDNA(
        num_questions=len(question_to_idx),
        num_skills=len(skill_to_idx),
        question_dim=QUESTION_DIM,
        skill_dim=SKILL_DIM,
        correctness_dim=CORRECTNESS_DIM,
        elapsed_dim=ELAPSED_DIM,
        attempt_dim=ATTEMPT_DIM,
        time_delta_dim=TIME_DELTA_DIM,
        model_dim=MODEL_DIM,
        num_heads=NUM_HEADS,
        num_layers=NUM_LAYERS,
        ff_dim=FF_DIM,
        dropout=dropout,
        max_seq_len=SEQ_LEN
    ).to(DEVICE)


def write_result_row(row):
    # Append one finished/failed experiment immediately.
    file_exists = RESULTS_CSV.exists()
    with open(RESULTS_CSV, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        if not file_exists or RESULTS_CSV.stat().st_size == 0:
            writer.writeheader()
        writer.writerow({k: row.get(k, "") for k in CSV_FIELDS})


def completed_experiment_ids():
    if not RESULTS_CSV.exists():
        return set()
    try:
        df = pd.read_csv(RESULTS_CSV)
        if not {"experiment_id", "status"}.issubset(df.columns):
            return set()
        return set(df.loc[df["status"].eq("completed"), "experiment_id"].astype(str))
    except Exception:
        return set()


def run_training_epoch(model, loader, optimizer, scaler, experiment_id, checkpoint_path):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss_sum = 0.0
    valid_targets_count = 0
    steps = 0
    samples = 0
    start = time.perf_counter()
    last_save = start

    for step, batch in enumerate(loader, start=1):
        batch = move_batch_to_device(batch)
        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            logits, _ = model(
                batch["question_idx"],
                batch["skill_idx"],
                batch["correctness_idx"],
                batch["elapsed"],
                batch["attempt_idx"],
                batch["time_delta"],
                batch["target_question_idx"]
            )
            mask = batch["target_mask"]
            if not mask.any():
                continue
            loss_matrix = F.binary_cross_entropy_with_logits(
                logits, batch["target"], reduction="none"
            )
            loss = loss_matrix[mask].mean()

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        count = int(mask.sum().item())
        running_loss_sum += float(loss.detach().item()) * count
        valid_targets_count += count
        steps += 1
        samples += int(batch["question_idx"].shape[0])

        if step % 100 == 0:
            elapsed = time.perf_counter() - start
            print(
                f"\r{experiment_id} | batch {step:,} | "
                f"loss {running_loss_sum / max(valid_targets_count, 1):.5f} | "
                f"{elapsed / 60:.1f} min",
                end=""
            )

        # Lightweight recovery checkpoint during long experiments.
        if steps % SAVE_CHECKPOINT_EVERY_STEPS == 0:
            torch.save({
                "experiment_id": experiment_id,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "training_steps": steps,
                "training_loss_so_far": running_loss_sum / max(valid_targets_count, 1),
                "config": {
                    "learning_rate": optimizer.param_groups[0]["lr"],
                    "dropout": getattr(model, "_tuning_dropout", None),
                    "weight_decay": optimizer.param_groups[0]["weight_decay"],
                    "seed": TUNING_SEED,
                },
                "checkpoint_type": "in_progress"
            }, checkpoint_path)

    elapsed = time.perf_counter() - start
    print()
    return {
        "training_loss": running_loss_sum / max(valid_targets_count, 1),
        "training_steps": steps,
        "training_samples": samples,
        "training_time_seconds": elapsed
    }


@torch.no_grad()
def evaluate_with_metrics(model, loader):
    model.eval()
    all_probs = []
    all_targets = []
    loss_sum = 0.0
    count = 0
    start = time.perf_counter()

    for batch in loader:
        batch = move_batch_to_device(batch)
        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            logits, _ = model(
                batch["question_idx"],
                batch["skill_idx"],
                batch["correctness_idx"],
                batch["elapsed"],
                batch["attempt_idx"],
                batch["time_delta"],
                batch["target_question_idx"]
            )

        mask = batch["target_mask"]
        if not mask.any():
            continue

        valid_logits = logits[mask].float()
        targets = batch["target"][mask].float()
        probs = torch.sigmoid(valid_logits)

        loss_sum += F.binary_cross_entropy_with_logits(
            valid_logits, targets, reduction="sum"
        ).item()
        count += targets.numel()
        all_probs.append(probs.detach().cpu().numpy())
        all_targets.append(targets.detach().cpu().numpy())

    validation_seconds = time.perf_counter() - start
    if not all_targets:
        raise RuntimeError("Validation produced zero target values.")

    y_true = np.concatenate(all_targets).astype(np.int32)
    y_prob = np.concatenate(all_probs).astype(np.float64)
    y_pred = (y_prob >= 0.5).astype(np.int32)

    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = float("nan")
    try:
        pr_auc = average_precision_score(y_true, y_prob)
    except ValueError:
        pr_auc = float("nan")

    return {
        "validation_loss": loss_sum / max(count, 1),
        "validation_accuracy": accuracy_score(y_true, y_pred),
        "roc_auc": auc,
        "pr_auc": pr_auc,
        "rmse": mean_squared_error(y_true, y_prob) ** 0.5,
        "log_loss": log_loss(y_true, np.clip(y_prob, 1e-7, 1 - 1e-7), labels=[0, 1]),
        "validation_time_seconds": validation_seconds,
        "validation_count": int(count)
    }


## Run the search

Run this cell once. It automatically proceeds through all pending configurations. If interrupted, rerun it: experiments already marked `completed` in the CSV are skipped. Each configuration gets its own checkpoint file.

The notebook saves the model/optimizer state periodically during training and a final checkpoint after evaluation. The in-progress checkpoint is for recovery/inspection; this version intentionally starts a fresh configuration when a configuration is rerun rather than silently resuming its weights into another hyperparameter setting.

In [15]:
def run_one_experiment(cfg):
    experiment_id = cfg["experiment_id"]
    checkpoint_path = CHECKPOINTS_DIR / f"{experiment_id}_checkpoint.pt"
    final_checkpoint_path = CHECKPOINTS_DIR / f"{experiment_id}_final.pt"
    started_at = datetime.now().isoformat(timespec="seconds")
    total_start = time.perf_counter()

    row = {
        **cfg,
        "seed": TUNING_SEED,
        "epochs_completed": 0,
        "training_steps": 0,
        "training_samples": 0,
        "train_parts": len(TUNING_PART_FILES),
        "validation_parts": len(TUNING_PART_FILES),
        "checkpoint_path": str(final_checkpoint_path),
        "status": "running",
        "error": "",
        "completed_at": ""
    }

    try:
        print("\\n" + "=" * 72)
        print(f"{experiment_id}: lr={cfg['learning_rate']}, dropout={cfg['dropout']}, weight_decay={cfg['weight_decay']}")
        print("=" * 72)

        seed_everything(TUNING_SEED)

        # Fresh model and optimizer for every configuration.
        model = create_tuning_model(cfg["dropout"])
        model._tuning_dropout = cfg["dropout"]

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=cfg["learning_rate"],
            weight_decay=cfg["weight_decay"]
        )
        
        # Same scheduler configuration as LearningDNA V3.
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=2,
            min_lr=1e-6
        )
        
        scaler = torch.amp.GradScaler(
            "cuda",
            enabled=USE_AMP
        )

        part_files_for_tuning = TUNING_PART_FILES
        train_dataset = KTSequenceDataset(
            TUNING_PART_FILES,
            train=True,
            seq_len=SEQ_LEN,
            stride=STRIDE,
            max_parts=None,
            start_part=0
        )
        val_dataset = KTSequenceDataset(
            TUNING_PART_FILES,
            train=False,
            seq_len=SEQ_LEN,
            stride=STRIDE,
            max_parts=None,
            start_part=0
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            collate_fn=collate_fn,
            num_workers=NUM_WORKERS,
            pin_memory=(DEVICE.type == "cuda")
        )
        val_loader = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            collate_fn=collate_fn,
            num_workers=NUM_WORKERS,
            pin_memory=(DEVICE.type == "cuda")
        )

        for epoch in range(1, TUNING_EPOCHS + 1):
            print(f"\n{experiment_id} | Epoch {epoch}/{TUNING_EPOCHS}")
        
            epoch_start = time.perf_counter()
        
            train_metrics = run_training_epoch(
                model,
                train_loader,
                optimizer,
                scaler,
                experiment_id,
                checkpoint_path
            )
        
            val_metrics = evaluate_with_metrics(model, val_loader)

            # Adjust learning rate using validation loss.
            scheduler.step(val_metrics["validation_loss"])
        
            epoch_seconds = time.perf_counter() - epoch_start
        
            # Update the running experiment information.
            row.update(train_metrics)
            row.update(val_metrics)
        
            row["epoch"] = epoch
            row["epochs_completed"] = epoch
        
            row["total_time_seconds"] = (
                time.perf_counter() - total_start
            )
        
            # Save a recovery checkpoint after every completed epoch.
            torch.save({
                "experiment_id": experiment_id,
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scaler_state": scaler.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "config": {
                    **cfg,
                    "seed": TUNING_SEED,
                    "seq_len": SEQ_LEN,
                    "stride": STRIDE,
                    "batch_size": BATCH_SIZE,
                    "max_parts": TUNING_MAX_PARTS,
                },
                "metrics": {
                    key: row.get(key)
                    for key in [
                        "training_loss",
                        "validation_loss",
                        "validation_accuracy",
                        "roc_auc",
                        "pr_auc",
                        "rmse",
                        "log_loss",
                    ]
                },
                "checkpoint_type": "epoch_complete",
            }, checkpoint_path)
        
            # Append this epoch's results immediately.
            row["status"] = "epoch_completed"
        
            row["completed_at"] = datetime.now().isoformat(
                timespec="seconds"
            )
        
            write_result_row(row)
        
            print(f"Training loss: {row['training_loss']:.5f}")
            print(f"Validation loss: {row['validation_loss']:.5f}")
            print(f"Accuracy: {row['validation_accuracy']:.5f}")
            print(f"ROC-AUC: {row['roc_auc']:.5f}")
            print(f"Epoch duration: {epoch_seconds / 60:.2f} min")

        total_seconds = time.perf_counter() - total_start
        row["total_time_seconds"] = total_seconds
        row["status"] = "completed"
        row["completed_at"] = datetime.now().isoformat(timespec="seconds")

        torch.save({
            "experiment_id": experiment_id,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "config": {
                **cfg,
                "seed": TUNING_SEED,
                "seq_len": SEQ_LEN,
                "stride": STRIDE,
                "batch_size": BATCH_SIZE,
                "max_parts": TUNING_MAX_PARTS,
            },
            "metrics": {k: row.get(k) for k in [
                "training_loss", "validation_loss", "validation_accuracy",
                "roc_auc", "pr_auc", "rmse", "log_loss",
                "training_time_seconds", "validation_time_seconds", "total_time_seconds"
            ]},
            "saved_at": row["completed_at"]
        }, final_checkpoint_path)

        # Keep CSV updated as soon as this configuration is finished.
        write_result_row(row)

        print(f"{experiment_id} completed.")
        print(f"  Train loss:       {row['training_loss']:.5f}")
        print(f"  Validation loss:  {row['validation_loss']:.5f}")
        print(f"  Accuracy:         {row['validation_accuracy']:.5f}")
        print(f"  ROC-AUC:          {row['roc_auc']:.5f}")
        print(f"  PR-AUC:           {row['pr_auc']:.5f}")
        print(f"  RMSE:             {row['rmse']:.5f}")
        print(f"  Log loss:         {row['log_loss']:.5f}")
        print(f"  Training time:    {row['training_time_seconds'] / 60:.2f} min")
        print(f"  Validation time:  {row['validation_time_seconds'] / 60:.2f} min")
        print(f"  Total time:       {row['total_time_seconds'] / 60:.2f} min")
        print(f"  Checkpoint:       {final_checkpoint_path}")

    except Exception as exc:
        row["status"] = "failed"
        row["error"] = repr(exc)
        row["total_time_seconds"] = time.perf_counter() - total_start
        row["completed_at"] = datetime.now().isoformat(timespec="seconds")
        write_result_row(row)
        print(f"ERROR in {experiment_id}: {exc}")
        print("Failure was logged to the results CSV; continuing to the next configuration.")

    finally:
        for name in [
            "model",
            "optimizer",
            "scheduler",
            "scaler",
            "train_loader",
            "val_loader",
            "train_dataset",
            "val_dataset"
        ]:
            if name in locals():
                try:
                    del locals()[name]
                except Exception:
                    pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


done_ids = completed_experiment_ids()
print("Already completed:", sorted(done_ids) if done_ids else "none")

for cfg in EXPERIMENTS:
    if cfg["experiment_id"] in done_ids:
        print(f"Skipping {cfg['experiment_id']} (already completed).")
        continue
    run_one_experiment(cfg)
    done_ids = completed_experiment_ids()

print("\\nHyperparameter search loop finished.")
print("Results:", RESULTS_CSV)


Already completed: ['N01', 'N02']
Skipping N01 (already completed).
Skipping N02 (already completed).
\n========================================================================
N03: lr=0.0003, dropout=0.05, weight_decay=5e-05


D:\Lokesh files\Projects\LearningDNA\venv\lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(



N03 | Epoch 1/5
N03 | batch 15,700 | loss 0.55250 | 47.3 min
Training loss: 0.55251
Validation loss: 0.54545
Accuracy: 0.72223
ROC-AUC: 0.75122
Epoch duration: 52.91 min

N03 | Epoch 2/5
N03 | batch 15,700 | loss 0.53637 | 40.7 min
Training loss: 0.53638
Validation loss: 0.54200
Accuracy: 0.72456
ROC-AUC: 0.75515
Epoch duration: 45.89 min

N03 | Epoch 3/5
N03 | batch 15,700 | loss 0.53427 | 37.5 min
Training loss: 0.53428
Validation loss: 0.53999
Accuracy: 0.72589
ROC-AUC: 0.75746
Epoch duration: 41.05 min

N03 | Epoch 4/5
N03 | batch 15,700 | loss 0.53275 | 38.6 min
Training loss: 0.53276
Validation loss: 0.53822
Accuracy: 0.72725
ROC-AUC: 0.75959
Epoch duration: 42.39 min

N03 | Epoch 5/5
N03 | batch 15,700 | loss 0.53149 | 60.0 min
Training loss: 0.53150
Validation loss: 0.53701
Accuracy: 0.72807
ROC-AUC: 0.76107
Epoch duration: 64.03 min
N03 completed.
  Train loss:       0.53150
  Validation loss:  0.53701
  Accuracy:         0.72807
  ROC-AUC:          0.76107
  PR-AUC:         

D:\Lokesh files\Projects\LearningDNA\venv\lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


N04 | batch 15,700 | loss 0.55404 | 38.0 min
Training loss: 0.55405
Validation loss: 0.54586
Accuracy: 0.72199
ROC-AUC: 0.75047
Epoch duration: 41.82 min

N04 | Epoch 2/5
N04 | batch 15,700 | loss 0.53759 | 38.1 min
Training loss: 0.53760
Validation loss: 0.54283
Accuracy: 0.72404
ROC-AUC: 0.75420
Epoch duration: 41.90 min

N04 | Epoch 3/5
N04 | batch 15,700 | loss 0.53581 | 38.3 min
Training loss: 0.53581
Validation loss: 0.54102
Accuracy: 0.72558
ROC-AUC: 0.75646
Epoch duration: 42.32 min

N04 | Epoch 4/5
N04 | batch 15,700 | loss 0.53448 | 39.0 min
Training loss: 0.53448
Validation loss: 0.53981
Accuracy: 0.72654
ROC-AUC: 0.75802
Epoch duration: 43.13 min

N04 | Epoch 5/5
N04 | batch 15,700 | loss 0.53333 | 44.1 min
Training loss: 0.53334
Validation loss: 0.53839
Accuracy: 0.72738
ROC-AUC: 0.75985
Epoch duration: 48.88 min
N04 completed.
  Train loss:       0.53334
  Validation loss:  0.53839
  Accuracy:         0.72738
  ROC-AUC:          0.75985
  PR-AUC:           0.85562
  RMSE:

## Compare and rank completed experiments

The ranking prioritizes ROC-AUC, then validation loss and RMSE as tie-breakers. Accuracy is shown but is not the only selection criterion. Because this is a one-epoch screening run on a subset, treat the ranking as preliminary.

In [17]:
if RESULTS_CSV.exists():
    results_df = pd.read_csv(RESULTS_CSV)
    completed_df = results_df[results_df["status"].eq("completed")].copy()

    if not completed_df.empty:
        completed_df = completed_df.sort_values(
            by=["roc_auc", "validation_loss", "rmse"],
            ascending=[False, True, True],
            na_position="last"
        ).reset_index(drop=True)
        completed_df.insert(0, "rank", np.arange(1, len(completed_df) + 1))
        completed_df.to_csv(RANKED_CSV, index=False)

        display_columns = [
            "rank", "experiment_id", "learning_rate", "dropout", "weight_decay",
            "validation_accuracy", "roc_auc", "pr_auc", "rmse", "log_loss",
            "training_time_seconds", "validation_time_seconds", "total_time_seconds"
        ]
        display(completed_df[[c for c in display_columns if c in completed_df.columns]])

        best = completed_df.iloc[0]
        print("\\nPreliminary best configuration:")
        print(f"  Experiment:   {best['experiment_id']}")
        print(f"  Learning rate:{best['learning_rate']}")
        print(f"  Dropout:      {best['dropout']}")
        print(f"  Weight decay: {best['weight_decay']}")
        print(f"  ROC-AUC:      {best['roc_auc']:.5f}")
        print(f"  Accuracy:     {best['validation_accuracy']:.5f}")
        print(f"  RMSE:         {best['rmse']:.5f}")
        print("\\nRanked CSV:", RANKED_CSV)
    else:
        print("No completed experiments yet. Run the search cell above.")
else:
    print("Results CSV does not exist yet. Run the search cell above.")

print("\n" + "=" * 60)
print("COMPARISON WITH LEARNINGDNA V3")
print("=" * 60)

print(
    f"V3 baseline ROC-AUC : "
    f"{V3_BASELINE['roc_auc']:.6f}"
)

print(
    f"V3 baseline Accuracy: "
    f"{V3_BASELINE['accuracy']:.6f}"
)

print(
    f"V3 baseline RMSE    : "
    f"{V3_BASELINE['rmse']:.6f}"
)

print("\nNew experiment results are based on the tuning subset.")
print("Final comparison requires full validation evaluation.")

,rank,experiment_id,learning_rate,dropout,weight_decay,validation_accuracy,roc_auc,pr_auc,rmse,log_loss,training_time_seconds,validation_time_seconds,total_time_seconds
0,1,N03,0.0003,0.05,0.00005,0.728072,0.761070,0.856663,0.425028,0.537011,3600.931175,235.451424,14794.189178
1,2,N02,0.0004,0.10,0.00005,0.727682,0.760294,0.856080,0.425340,0.537713,2287.393445,220.485603,14273.179623
2,3,N04,0.0003,0.15,0.00005,0.727378,0.759855,0.855617,0.425605,0.538387,2649.965284,270.441748,13083.454094
3,4,N01,0.0002,0.10,0.00005,0.727064,0.759732,0.855758,0.425610,0.538265,2288.638916,220.961149,13252.654499


\nPreliminary best configuration:
  Experiment:   N03
  Learning rate:0.0003
  Dropout:      0.05
  Weight decay: 5e-05
  ROC-AUC:      0.76107
  Accuracy:     0.72807
  RMSE:         0.42503
\nRanked CSV: D:\Lokesh files\Projects\LearningDNA\models\learningdna_v2\hyperparameter_tuning_v3\hyperparameter_search_ranked.csv

COMPARISON WITH LEARNINGDNA V3
V3 baseline ROC-AUC : 0.771046
V3 baseline Accuracy: 0.735386
V3 baseline RMSE    : 0.420077

New experiment results are based on the tuning subset.
Final comparison requires full validation evaluation.


## Notes before using the winner

- `hyperparameter_search_results.csv` is the append-only experiment log; each completed or failed configuration is recorded immediately.
- `hyperparameter_search_ranked.csv` is regenerated by the comparison cell.
- Each experiment uses a fresh model and optimizer. No experiment inherits another experiment's weights.
- Checkpoints are stored separately under `hyperparameter_tuning/checkpoints/`.
- This notebook tests only the specified hyperparameters. Architecture, data split rule, sequence length, stride, batch size, loss function, and existing data pipeline are retained from the original notebook.
- After selecting a candidate, verify it with multiple epochs and the full validation set before using it for the final result.
